# ============================================================================

# [1] IMPORTS & SETUP
# ============================================================================

In [21]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import os
from dotenv import load_dotenv
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()

print("="*80)
print("PROJECT 1: 4.11 - COURSES AND SECTIONS TOGETHER ASSIGNMENT")
print("="*80)





PROJECT 1: 4.11 - COURSES AND SECTIONS TOGETHER ASSIGNMENT


# ============================================================================
# [2] LOAD CSV DATA
# ============================================================================

In [22]:
print("\n[STEP 1] Loading CSV data...")

try:
    df = pd.read_csv("365 Courses data.csv", encoding='latin-1')
    print(f"✓ Loaded {len(df)} rows")
    print(f"✓ Columns: {df.columns.tolist()}")
except FileNotFoundError:
    print("✗ CSV file not found. Make sure '365 Courses data.csv' is in the same directory")
    raise
except Exception as e:
    print(f"✗ Error loading CSV: {e}")
    raise

# Display sample data
print("\nFirst 2 rows:")
print(df.head(2))

# Data quality check
print(f"\nData Quality:")
print(f"  Total rows: {len(df)}")
print(f"  Unique course_ids: {df['course_id'].nunique() if 'course_id' in df.columns else 'N/A'}")
print(f"  Null values:\n{df.isnull().sum()}")






[STEP 1] Loading CSV data...
✓ Loaded 106 rows
✓ Columns: ['course_name', 'course_slug', 'course_technology', 'course_description', 'course_topic', 'course_description_short']

First 2 rows:
                                         course_name         course_slug  \
0                            Introduction to Tableau             tableau   
1  The Complete Data Visualization Course with Py...  data-visualization   

  course_technology                                 course_description  \
0           tableau  Tableau is now one of the most popular busines...   
1            python  The Data Visualization course is designed for ...   

         course_topic                           course_description_short  
0  data visualization  Teaching you how to tell compelling stories wi...  
1  data visualization  Teaching you how to master the art of creating...  

Data Quality:
  Total rows: 106
  Unique course_ids: N/A
  Null values:
course_name                 0
course_slug                 

there is no course id here in the dataset so we need to use another method to have it. Two alternatives are possible: generate it randomly or use slug. 

In [23]:
df.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...


In [24]:
df["course_slug"].duplicated().sum()

np.int64(1)

In [25]:
duplicates = df[df['course_slug'].duplicated(keep=False)]
print(duplicates[['course_slug']])

                           course_slug
60  learn-machine-learning-process-a-z
61  learn-machine-learning-process-a-z


In [26]:
df = df.drop_duplicates()

In [27]:
df["course_slug"].duplicated().sum()

np.int64(1)

In [28]:
df[df['course_slug'].duplicated(keep=False)].sort_values('course_slug')

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
60,The Machine Learning Process A-Z,learn-machine-learning-process-a-z,python,Data science education focuses too much on the...,machine learning,There is so much more to data science than jus...
61,The Machine Learning Process A-Z,learn-machine-learning-process-a-z,python,Data science education focuses too much on the...,machine learning,Learn what goes on beneath the surface of mach...


AS Slug is not duplicated but not a ligne we should now generate an artificial unique key

In [29]:
df['course_id'] = df.index

there is also no section-id and section_decription in the dataset , so I need to create them

In [30]:
sections_df = (
    df.groupby('course_topic')['course_description_short']
    .apply(lambda x: ' '.join(x.unique()))
    .reset_index()
)

sections_df['section_id'] = range(1, len(sections_df) + 1)

sections_df.rename(columns={
    'course_topic': 'section_name',
    'course_description_short': 'section_description'
}, inplace=True)

In [31]:
sections_df.head()

,section_name,section_description,section_id
0,a/b testing,A world-class professional teaches you how to ...,1
1,business analytics,Giving you the knowledge to leverage the value...,2
2,business fundamentals,A foundational business course that teaches yo...,3
3,career development,A data scientists guide to landing a data sci...,4
4,data analysis,Learn how to work with pivot tables and create...,5


In [32]:
# join the section table to the original dataframe
df = df.merge(sections_df[['section_id', 'section_name','section_description']], left_on='course_topic', right_on='section_name', how='left')

In [33]:
df.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short,course_id,section_id,section_name,section_description
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...,0,7,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...,1,7,data visualization,Teaching you how to tell compelling stories wi...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a...",2,12,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...,3,6,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...,4,10,machine learning,Introducing you to the field of data science a...


In [42]:
# save the updated dataframes to new CSV files
df.to_csv("courses_with_sections.csv", index=False)

# ============================================================================
# [3] VALIDATE COLUMNS
# ============================================================================

In [ ]:
print("\n[STEP 2] Validating columns...")

# Check if required columns exist
required_columns = ['course_id', 'section_id', 'course_name', 'section_name', 'course_description']
available_columns = df.columns.tolist()

missing = [col for col in required_columns if col not in available_columns]
if missing:
    print(f" WARNING: Missing columns: {missing}")
    print(f"Available columns: {available_columns}")
else:
    print(f"✓ All required columns present")






[STEP 2] Validating columns...
✓ All required columns present


# ============================================================================
# [4] INITIALIZE MODEL
# ============================================================================

In [35]:
print("\n[STEP 3] Initializing SentenceTransformer model...")

model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dimension = model.get_sentence_embedding_dimension()

print(f"✓ Model loaded: all-MiniLM-L6-v2")
print(f"✓ Embedding dimension: {embedding_dimension}")





2026-06-11 02:16:14,376 - INFO - No device provided, using cpu



[STEP 3] Initializing SentenceTransformer model...


2026-06-11 02:16:18,114 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 02:16:18,277 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-11 02:16:18,707 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 02:16:18,871 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-11 02:16:18,880 - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-06-11 02:16:19,293 - INFO - HTTP Request: HEAD https://huggingface.co/s

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-06-11 02:16:23,618 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-11 02:16:24,001 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-11 02:16:24,433 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-11 02:16:24,818 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-11 02:16:25,260 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-11 02:16:25,816 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

✓ Model loaded: all-MiniLM-L6-v2
✓ Embedding dimension: 384


C:\Users\fredb\AppData\Local\Temp\ipykernel_8332\1922687550.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


# ============================================================================
# [5] DEFINE WEIGHTS
# ============================================================================

In [36]:
print("\n[STEP 4] Defining weights...")

# Weights for different fields
weights = {
    'course_name': 5,
    'section_names': 4,
    'course_description': 3,
    'course_topic': 2,
    'course_technology': 1
}

total_weight = sum(weights.values())

print(f"Weights:")
for field, weight in weights.items():
    print(f"  {field}: {weight}")
print(f"Total weight: {total_weight}")






[STEP 4] Defining weights...
Weights:
  course_name: 5
  section_names: 4
  course_description: 3
  course_topic: 2
  course_technology: 1
Total weight: 15


# ============================================================================
# [6] HELPER FUNCTION - SAFE ENCODING
# ============================================================================

In [37]:
def safe_encode(text, weight=1.0):
    """
    Safely encode text, handling None/empty values
    
    Args:
        text: Text to encode (can be None, empty string, list, etc.)
        weight: Weight multiplier for the embedding
        
    Returns:
        Weighted embedding vector
    """
    # Handle None or empty
    if text is None or (isinstance(text, str) and not text.strip()):
        return np.zeros(embedding_dimension)
    
    # Handle lists (join them)
    if isinstance(text, list):
        text = ' '.join([str(t) for t in text if t])
        if not text.strip():
            return np.zeros(embedding_dimension)
    
    try:
        embedding = model.encode(str(text), show_progress_bar=False)
        return embedding * weight
    except Exception as e:
        logger.warning(f"Encoding error for '{str(text)[:30]}...': {e}")
        return np.zeros(embedding_dimension)

# Test safe_encode
print("\n[STEP 5] Testing safe_encode function...")
test_embedding = safe_encode("test text", 1.0)
print(f"✓ Test embedding shape: {test_embedding.shape}")
print(f"✓ Test embedding dimension matches: {len(test_embedding) == embedding_dimension}")






[STEP 5] Testing safe_encode function...
✓ Test embedding shape: (384,)
✓ Test embedding dimension matches: True


# ============================================================================
# [7] AGGREGATE DATA BY COURSE
# ============================================================================

In [38]:
print("\n[STEP 6] Aggregating sections by course...")

# Group by course_id and aggregate
agg_dict = {
    'course_name': 'first',
    'course_slug': 'first',
    'course_description': 'first',
    'course_description_short': 'first',
    'course_technology': 'first',
    'course_topic': 'first', 
    'course_instructor_quote': 'first',
    'section_name': lambda x: list(x),  # Collect all section names
    'section_id': 'count'  # Count sections
}

# Only aggregate columns that exist
available_agg = {k: v for k, v in agg_dict.items() if k in df.columns}

course_agg = df.groupby('course_id').agg(available_agg)
course_agg = course_agg.rename(columns={'section_id': 'num_sections'})

print(f"✓ Aggregated to {len(course_agg)} unique courses")
print(f"✓ Total sections preserved: {course_agg['num_sections'].sum()}")
print(f"\nSample aggregated course:")
print(course_agg.iloc[0])





[STEP 6] Aggregating sections by course...
✓ Aggregated to 106 unique courses
✓ Total sections preserved: 106

Sample aggregated course:
course_name                                           Introduction to Tableau
course_slug                                                           tableau
course_description          Tableau is now one of the most popular busines...
course_description_short    Teaching you how to tell compelling stories wi...
course_technology                                                     tableau
course_topic                                               data visualization
section_name                                             [data visualization]
num_sections                                                                1
Name: 0, dtype: object



# ============================================================================
# [8] CREATE COURSE EMBEDDINGS
# ============================================================================

In [ ]:
print("\n[STEP 7] Creating weighted embeddings for each course...")

course_embeddings = []
errors = []

for course_id, row in course_agg.iterrows():
    try:
        # Encode each field with its weight
        emb_course_name = safe_encode(row.get('course_name', ''), weights['course_name'])
        
        # Section names as aggregated list
        section_text = ' '.join(row.get('section_name', [])) if isinstance(row.get('section_name'), list) else str(row.get('section_name', ''))
        emb_sections = safe_encode(section_text, weights['section_names'])
        
        emb_description = safe_encode(row.get('course_description', ''), weights['course_description'])
        emb_topic = safe_encode(row.get('course_topic', ''), weights['course_topic'])
        emb_technology = safe_encode(row.get('course_technology', ''), weights['course_technology'])
        
        # Composite embedding: weighted sum normalized by total weight
        composite = (emb_course_name + emb_sections + emb_description + emb_topic + emb_technology) / total_weight
        
        # Validation
        assert len(composite) == embedding_dimension, f"Dimension mismatch for course {course_id}"
        assert not np.isnan(composite).any(), f"NaN values in course {course_id}"
        assert not np.isinf(composite).any(), f"Inf values in course {course_id}"
        
        # Store result
        course_embeddings.append({
            'course_id': int(course_id),
            'embedding': composite.tolist(),
            'metadata': {
                'course_name': str(row.get('course_name', 'Unknown')),
                'course_topic': str(row.get('course_topic', '')),
                'course_technology': str(row.get('course_technology', '')),
                'num_sections': int(row.get('num_sections', 0))
            }
        })
    
    except Exception as e:
        logger.warning(f"Error processing course {course_id}: {e}")
        errors.append((course_id, str(e)))
        continue

print(f"✓ Generated {len(course_embeddings)} course embeddings")
if errors:
    print(f" {len(errors)} errors during generation:")
    for course_id, error in errors[:5]:
        print(f"    Course {course_id}: {error}")

# Verify embedding quality
print(f"\nEmbedding Quality Check:")
embedding_norms = [np.linalg.norm(np.array(e['embedding'])) for e in course_embeddings]
print(f"  Min norm: {min(embedding_norms):.4f}")
print(f"  Max norm: {max(embedding_norms):.4f}")
print(f"  Mean norm: {np.mean(embedding_norms):.4f}")






[STEP 7] Creating weighted embeddings for each course...
✓ Generated 106 course embeddings

Embedding Quality Check:
  Min norm: 0.6879
  Max norm: 0.8822
  Mean norm: 0.7814


# ============================================================================
# [9] INITIALIZE PINECONE
# ============================================================================

In [47]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override = True)


True

In [48]:
pc = Pinecone(api_key = os.environ.get("API_KEY"))

In [49]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [50]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")




2026-06-11 02:39:29,013 - INFO - Listing indexes
2026-06-11 02:39:30,918 - INFO - HTTP Request: GET https://api.pinecone.io/indexes "HTTP/1.1 200 OK"


my-index not in index list.


In [52]:
from pinecone import Pinecone, ServerlessSpec

In [53]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

2026-06-11 02:40:00,231 - INFO - Creating index 'my-index'
2026-06-11 02:40:02,550 - INFO - Describing index 'my-index'
2026-06-11 02:40:02,837 - INFO - HTTP Request: GET https://api.pinecone.io/indexes/my-index "HTTP/1.1 200 OK"
2026-06-11 02:40:07,840 - INFO - Describing index 'my-index'
2026-06-11 02:40:10,627 - INFO - HTTP Request: GET https://api.pinecone.io/indexes/my-index "HTTP/1.1 200 OK"


IndexModel(name='my-index', metric='cosine', host='https://my-index-vzjiolx.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [54]:
index = pc.Index(index_name)

2026-06-11 02:40:34,639 - INFO - Index client created for host https://my-index-vzjiolx.svc.aped-4627-b74a.pinecone.io


# ============================================================================
# [10] UPSERT VECTORS TO PINECONE
# ============================================================================

In [55]:
print("\n[STEP 9] Upserting vectors to Pinecone...")

# Prepare vectors in correct format: (id, embedding, metadata)
vectors_to_upsert = [
    (str(item['course_id']), item['embedding'], item['metadata'])
    for item in course_embeddings
]

print(f"Preparing {len(vectors_to_upsert)} vectors...")

try:
    index.upsert(vectors=vectors_to_upsert)
    print(f"✓ Successfully upserted {len(vectors_to_upsert)} vectors to Pinecone")
except Exception as e:
    print(f"✗ Upsert failed: {e}")
    raise





2026-06-11 02:40:56,574 - INFO - Upserting 106 vectors into namespace ''



[STEP 9] Upserting vectors to Pinecone...
Preparing 106 vectors...
✓ Successfully upserted 106 vectors to Pinecone


# ============================================================================
# [11] SEARCH FUNCTION
# ============================================================================

In [56]:
print("\n[STEP 10] Setting up search function...")

def semantic_search(query, top_k=10, threshold=0.3):
    """
    Execute semantic search on courses
    
    Args:
        query (str): Search query
        top_k (int): Number of results to retrieve
        threshold (float): Minimum similarity score
        
    Returns:
        list: List of matching courses with scores
    """
    try:
        # Encode query
        query_embedding = model.encode(query, show_progress_bar=False).tolist()
        
        # Query Pinecone
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )
        
        # Filter by threshold and format results
        filtered_results = []
        for match in results['matches']:
            if match['score'] >= threshold:
                filtered_results.append({
                    'course_id': match['id'],
                    'score': match['score'],
                    'name': match['metadata']['course_name'],
                    'topic': match['metadata']['course_topic'],
                    'technology': match['metadata']['course_technology'],
                    'sections': match['metadata']['num_sections']
                })
        
        return filtered_results
    
    except Exception as e:
        logger.error(f"Search error: {e}")
        return []

print(f"✓ Search function ready")






[STEP 10] Setting up search function...
✓ Search function ready


# ============================================================================
# [12] TEST SEARCH
# ============================================================================

In [57]:
print("\n" + "="*80)
print("TESTING SEMANTIC SEARCH")
print("="*80)

# Test queries
test_queries = [
    ("machine learning", 0.3),
    ("data visualization", 0.3),
    ("python programming", 0.3),
    ("clustering algorithms", 0.3)
]

for query, threshold in test_queries:
    print(f"\n{'─'*80}")
    print(f"Query: '{query}' (threshold: {threshold})")
    print(f"{'─'*80}")
    
    results = semantic_search(query, top_k=5, threshold=threshold)
    
    if not results:
        print("No results found")
        continue
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['name']}")
        print(f"   Course ID: {result['course_id']}")
        print(f"   Topic: {result['topic']}")
        print(f"   Technology: {result['technology']}")
        print(f"   Sections: {result['sections']}")
        print(f"   Score: {result['score']:.4f}")

print("\n" + "="*80)
print("✓ PROJECT 1 COMPLETE - Semantic search engine for courses is ready")
print("="*80)



2026-06-11 02:41:12,871 - INFO - Querying index with top_k=5



TESTING SEMANTIC SEARCH

────────────────────────────────────────────────────────────────────────────────
Query: 'machine learning' (threshold: 0.3)
────────────────────────────────────────────────────────────────────────────────


2026-06-11 02:41:14,807 - INFO - Querying index with top_k=5



1. Machine Learning in Excel
   Course ID: 40
   Topic: machine learning
   Technology: excel
   Sections: 1
   Score: 0.8936

2. The Machine Learning Process A-Z
   Course ID: 61
   Topic: machine learning
   Technology: python
   Sections: 1
   Score: 0.8906

3. The Machine Learning Process A-Z
   Course ID: 60
   Topic: machine learning
   Technology: python
   Sections: 1
   Score: 0.8906

4. Machine Learning with Support Vector Machines
   Course ID: 41
   Topic: machine learning
   Technology: python
   Sections: 1
   Score: 0.8848

5. The Machine Learning Algorithms A-Z
   Course ID: 89
   Topic: machine learning
   Technology: python
   Sections: 1
   Score: 0.9182

────────────────────────────────────────────────────────────────────────────────
Query: 'data visualization' (threshold: 0.3)
────────────────────────────────────────────────────────────────────────────────


2026-06-11 02:41:15,542 - INFO - Querying index with top_k=5



1. The Complete Data Visualization Course with Python, R, Tableau, and Excel
   Course ID: 1
   Topic: data visualization
   Technology: python
   Sections: 1
   Score: 0.8936

2. Growth Analysis with SQL, Python, and Tableau  
   Course ID: 105
   Topic: data visualization
   Technology: tableau
   Sections: 1
   Score: 0.7701

3. Introduction to Tableau
   Course ID: 0
   Topic: data visualization
   Technology: tableau
   Sections: 1
   Score: 0.7648

4. Customer Churn Analysis with SQL and Tableau
   Course ID: 90
   Topic: data visualization
   Technology: tableau
   Sections: 1
   Score: 0.7388

5. Intro to PowerPoint
   Course ID: 95
   Topic: data visualization
   Technology: powerpoint
   Sections: 1
   Score: 0.7195

────────────────────────────────────────────────────────────────────────────────
Query: 'python programming' (threshold: 0.3)
────────────────────────────────────────────────────────────────────────────────


2026-06-11 02:41:16,232 - INFO - Querying index with top_k=5



1. Introduction to Python
   Course ID: 27
   Topic: programming
   Technology: python
   Sections: 1
   Score: 0.8966

2. Intermediate Python Programming
   Course ID: 63
   Topic: programming
   Technology: python
   Sections: 1
   Score: 0.8966

3. Machine Learning in Python
   Course ID: 29
   Topic: programming
   Technology: python
   Sections: 1
   Score: 0.8353

4. Working with Text Files in Python
   Course ID: 62
   Topic: programming
   Technology: python
   Sections: 1
   Score: 0.8221

5. Python Programmer Bootcamp
   Course ID: 10
   Topic: programming
   Technology: python
   Sections: 1
   Score: 0.7985

────────────────────────────────────────────────────────────────────────────────
Query: 'clustering algorithms' (threshold: 0.3)
────────────────────────────────────────────────────────────────────────────────

1. Machine Learning with K-Nearest Neighbors
   Course ID: 44
   Topic: machine learning
   Technology: python
   Sections: 1
   Score: 0.5558

2. Machine Learn